In [7]:
# Question 6.1 One large P1 test sample from the three files on Canvas
import string

files = ['P1_test1.txt', 'P1_test2.txt', 'P1_test3.txt']

def preprocess(text):
    """Split on whitespace, lowercase, and strip punctuation from each token."""
    tokens = text.split()                                         # 1. Split on whitespace
    tokens = [t.lower() for t in tokens]                         # 2. Lowercase
    tokens = [t.strip(string.punctuation) for t in tokens]       # 3. Strip punctuation
    tokens = [t for t in tokens if t]                            # Remove any empty strings
    tokens = [t for t in tokens if len(t) >= 4]
    return tokens

results = {}

for path in files:
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    tokens = preprocess(text)
    results[path] = tokens

# --- Word Counts ---
for path, tokens in results.items():
    print(f"{path}: {len(tokens)} words")

P1_test1.txt: 71 words
P1_test2.txt: 109 words
P1_test3.txt: 60 words


In [8]:
# Question 6.2 Making a prediction on the sample using Naive Bayes
from collections import Counter

combined_tokens = [token for tokens in results.values() for token in tokens]
total_words = len(combined_tokens)
word_counts = Counter(combined_tokens)

proportional_probs = {word: count / total_words for word, count in word_counts.items()}

# Sort by probability descending for readability
proportional_probs = dict(sorted(proportional_probs.items(), key=lambda x: x[1], reverse=True))

print(f"\nTotal words (combined): {total_words}")
print(f"\n{'Word':<20} {'Count':>6} {'Probability':>12}")
print("-" * 40)
for word, prob in proportional_probs.items():
    print(f"{word:<20} {word_counts[word]:>6} {prob:>12.4f}")

# recision problem — most words appear only once or twice, giving them very small probabilities.
# If you were to multiply many of these small values together (as Naive Bayes does), the product
# would quickly approach zero and risk underflowing.



Total words (combined): 240

Word                  Count  Probability
----------------------------------------
model                    13       0.0542
predictor                11       0.0458
with                      7       0.0292
your                      7       0.0292
variable                  6       0.0250
each                      6       0.0250
data                      6       0.0250
values                    6       0.0250
training                  5       0.0208
features                  5       0.0208
linear                    4       0.0167
this                      4       0.0167
that                      4       0.0167
used                      4       0.0167
variables                 3       0.0125
submit                    3       0.0125
gradescope                3       0.0125
regression                3       0.0125
would                     3       0.0125
response                  3       0.0125
change                    3       0.0125
code                      3

In [9]:
import math

# --- Class Priors: P(c) ---
# Each file's share of the total 240 words, log transformed
log_priors = {
    path: math.log(len(tokens) / total_words)
    for path, tokens in results.items()
}

# --- Conditional Probabilities: P(w | c) ---
# For each file, count words then divide by that file's total word count
log_conditionals = {}
for path, tokens in results.items():
    file_total = len(tokens)
    file_counts = Counter(tokens)
    log_conditionals[path] = {
        word: math.log(count / file_total)
        for word, count in file_counts.items()
    }

# --- Output ---
print(f"{'File':<20} {'Word Count':>10} {'Log Prior':>12}")
print("-" * 44)
for path in files:
    print(f"{path:<20} {len(results[path]):>10} {log_priors[path]:>12.4f}")

print(f"\n{'Word':<20} " + " ".join(f"{p:>14}" for p in files))
print("-" * (20 + 15 * len(files)))

all_words = set(word for tokens in results.values() for word in tokens)
for word in sorted(all_words):
    row = f"{word:<20} "
    row += " ".join(
        f"{log_conditionals[path].get(word, float('-inf')):>14.4f}"
        for path in files
    )
    print(row)

File                 Word Count    Log Prior
--------------------------------------------
P1_test1.txt                 71      -1.2180
P1_test2.txt                109      -0.7893
P1_test3.txt                 60      -1.3863

Word                   P1_test1.txt   P1_test2.txt   P1_test3.txt
-----------------------------------------------------------------
1-mse                          -inf        -4.6913           -inf
achieve                        -inf        -4.6913           -inf
after                          -inf           -inf        -4.0943
analysis                       -inf           -inf        -4.0943
answers                     -4.2627           -inf           -inf
apply                          -inf        -3.9982           -inf
around                         -inf        -4.6913           -inf
between                     -4.2627           -inf           -inf
brief                       -4.2627           -inf           -inf
calculating                    -inf        -4.69

In [10]:
import math

# --- Compute log score per class using combined_tokens as the test sample ---
log_scores = {}

for path in files:
    # Start with the log prior for this class
    score = log_priors[path]

    for word in combined_tokens:
        if word in log_conditionals[path]:
            score += log_conditionals[path][word]
        # Unknown words are skipped to avoid -inf dominating

    log_scores[path] = score

# --- Report results ---
print(f"Test sample word count: {len(combined_tokens)}")
print(f"\n{'File':<20} {'Log Score':>12}")
print("-" * 34)
for path, score in log_scores.items():
    print(f"{path:<20} {score:>12.4f}")

# --- Final prediction ---
predicted_class = max(log_scores, key=log_scores.get)
print(f"\nFinal Prediction: {predicted_class}")

Test sample word count: 240

File                    Log Score
----------------------------------
P1_test1.txt            -440.3654
P1_test2.txt            -618.3463
P1_test3.txt            -491.6716

Final Prediction: P1_test1.txt
